# Model estimation and forecasting

## Define paths, returns, samples, and proxy for volatility

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_PATH = Path("../data_clean/btc_daily_returns.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.sort_values("Date")
df = df.set_index("Date")

df.head()

,price,log_return
Date,,
2016-01-02,433.437988,-0.206512
2016-01-03,430.010986,-0.793798
2016-01-04,433.091003,0.713712
2016-01-05,431.959991,-0.261490
2016-01-06,429.105011,-0.663130


In [5]:
print(df.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 3803 entries, 2016-01-02 to 2026-05-31
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   price       3803 non-null   float64
 1   log_return  3803 non-null   float64
dtypes: float64(2)
memory usage: 89.1 KB
None


In [6]:
returns = df["log_return"].dropna()
returns.name = "BTC daily log return (%)"

returns.head()

Date
2016-01-02   -0.206512
2016-01-03   -0.793798
2016-01-04    0.713712
2016-01-05   -0.261490
2016-01-06   -0.663130
Name: BTC daily log return (%), dtype: float64

In [7]:
OUT_OF_SAMPLE_FRACTION = 0.20

n_obs = len(returns)
split_idx = int(np.floor((1 - OUT_OF_SAMPLE_FRACTION) * n_obs))

returns_in = returns.iloc[:split_idx].copy()
returns_out = returns.iloc[split_idx:].copy()

print(f"Total number of return observations: {n_obs}")
print(f"In-sample observations:             {len(returns_in)}")
print(f"Out-of-sample observations:         {len(returns_out)}")
print()
print(f"In-sample period:     {returns_in.index.min().date()} to {returns_in.index.max().date()}")
print(f"Out-of-sample period: {returns_out.index.min().date()} to {returns_out.index.max().date()}")

Total number of return observations: 3803
In-sample observations:             3042
Out-of-sample observations:         761

In-sample period:     2016-01-02 to 2024-04-30
Out-of-sample period: 2024-05-01 to 2026-05-31


In [10]:
# ------------------------------------------------------------
# Construct the out-of-sample realised volatility proxy
# ------------------------------------------------------------

mean_return_in = returns_in.mean()

volatility_proxy_out = (returns_out - mean_return_in) ** 2
volatility_proxy_out.name = "realised_volatility_proxy"

print(f"In-sample mean return: {mean_return_in:.6f}")
print()
print(volatility_proxy_out.head())

In-sample mean return: 0.162355

Date
2024-05-01    17.400140
2024-05-02     1.739983
2024-05-03    36.160564
2024-05-04     2.010098
2024-05-05     0.003136
Name: realised_volatility_proxy, dtype: float64


## Estimate ARCH(1)-normal benchmark model

In [ ]:
# imprt required packages
import arch
from arch import arch_model

## Diagnostic tests for ARCH(1)-Normal

In [29]:
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

In [30]:
DIAGNOSTIC_LAGS = [10, 20, 30]
ARCH_LM_LAGS = 10

std_resid = arch1_normal_fitted["standardized_residual"].dropna()
std_resid_sq = std_resid ** 2

std_resid.name = "standardized_residual"
std_resid_sq.name = "squared_standardized_residual"

print(f"Number of standardized residuals used for diagnostics: {len(std_resid)}")

Number of standardized residuals used for diagnostics: 3042


In [35]:
#Ljung-box test on standardized residuals
lb_std_resid = acorr_ljungbox(
    std_resid,
    lags=DIAGNOSTIC_LAGS,
    return_df=True
)

lb_std_resid = lb_std_resid.rename(columns={
    "lb_stat": "lb_stat_standardized_residual",
    "lb_pvalue": "lb_pvalue_standardized_residual"
})


#Ljung-Box test on squared standardized residuals
lb_std_resid_sq = acorr_ljungbox(
    std_resid_sq,
    lags=DIAGNOSTIC_LAGS,
    return_df=True
)

lb_std_resid_sq = lb_std_resid_sq.rename(columns={
    "lb_stat": "lb_stat_squared_standardized_residual",
    "lb_pvalue": "lb_pvalue_squared_standardized_residual"
})


#Combine diagnostic results
ljung_box_diagnostics_arch1_normal = pd.concat(
    [lb_std_resid, lb_std_resid_sq],
    axis=1
)

ljung_box_diagnostics_arch1_normal.index.name = "lag"

ljung_box_diagnostics_arch1_normal

,lb_stat_standardized_residual,lb_pvalue_standardized_residual,lb_stat_squared_standardized_residual,lb_pvalue_squared_standardized_residual
lag,,,,
10,13.016149,0.222770,58.423830,7.192892e-09
20,20.445164,0.430410,70.749570,1.373450e-07
30,32.874936,0.327995,79.463223,2.359155e-06


In [36]:
# ARCH-LM test for remaining ARCH effects

arch_lm_stat, arch_lm_pvalue, arch_lm_f_stat, arch_lm_f_pvalue = het_arch(
    std_resid,
    nlags=ARCH_LM_LAGS,
    ddof=arch1_normal_result.num_params
)

arch_lm_arch1_normal = pd.Series({
    "model": "ARCH(1)",
    "distribution": "Normal",
    "lags": ARCH_LM_LAGS,
    "lm_stat": arch_lm_stat,
    "lm_pvalue": arch_lm_pvalue,
    "f_stat": arch_lm_f_stat,
    "f_pvalue": arch_lm_f_pvalue
})

arch_lm_arch1_normal

model             ARCH(1)
distribution       Normal
lags                   10
lm_stat         53.991481
lm_pvalue             0.0
f_stat           5.482615
f_pvalue              0.0
dtype: object

## Generalized model estimation workflow of ARCH-family models

In [39]:
def fit_volatility_model(
    returns_series,
    model_name,
    volatility_model,
    p,
    o=0,
    q=0,
    distribution="normal",
    mean_model="Constant",
    rescale=False,
    cov_type="classic"
):
    """
    Estimate one ARCH-family volatility model.

    Parameters
    ----------
    returns_series : pandas.Series
        The return series used for model estimation.
    model_name : str
        A readable model name, for example "ARCH(1)" or "GARCH(1,1)".
    volatility_model : str
        Volatility model type used by the arch package, for example
        "ARCH", "GARCH", or "EGARCH".
    p : int
        Number of ARCH terms.
    o : int, optional
        Number of asymmetric terms. This is relevant for EGARCH.
    q : int, optional
        Number of lagged conditional variance terms.
    distribution : str, optional
        Innovation distribution used by the arch package.
        Examples: "normal", "t", "ged", "skewt".
    mean_model : str, optional
        Conditional mean specification. We use "Constant".
    rescale : bool, optional
        Whether the arch package may internally rescale the data.
        We use False because returns are already expressed in percentages.
    cov_type : str, optional
        Type of covariance estimator for standard errors.
        We use "classic" to match standard maximum likelihood inference.

    Returns
    -------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.
    """

    model = arch_model(
        y=returns_series,
        mean=mean_model,
        vol=volatility_model,
        p=p,
        o=o,
        q=q,
        dist=distribution,
        rescale=rescale
    )

    result = model.fit(
        disp="off",
        cov_type=cov_type
    )

    # Store readable metadata directly on the result object.
    # This is not used by the arch package itself, but it helps us later
    # when collecting outputs from many different models.
    result.model_name = model_name
    result.distribution_name = distribution
    result.volatility_model_name = volatility_model

    return result

In [40]:
def extract_parameter_table(result):
    """
    Extract parameter estimates, standard errors, t-values and p-values
    from an estimated ARCH-family model.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.

    Returns
    -------
    parameter_table : pandas.DataFrame
        Table with one row per estimated parameter.
    """

    parameter_table = pd.DataFrame({
        "model": result.model_name,
        "distribution": result.distribution_name,
        "parameter": result.params.index,
        "estimate": result.params.values,
        "std_error": result.std_err.values,
        "t_value": result.tvalues.values,
        "p_value": result.pvalues.values
    })

    return parameter_table

In [41]:
def extract_model_info(result):
    """
    Extract log-likelihood, AIC, BIC and related model information.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.

    Returns
    -------
    model_info : pandas.Series
        Summary information for one estimated model.
    """

    model_info = pd.Series({
        "model": result.model_name,
        "distribution": result.distribution_name,
        "log_likelihood": result.loglikelihood,
        "aic": result.aic,
        "bic": result.bic,
        "n_obs": result.nobs,
        "num_params": result.num_params,
        "convergence_flag": result.convergence_flag
    })

    return model_info

In [42]:
def extract_fitted_series(result, returns_series):
    """
    Extract residuals, fitted conditional volatility, conditional variance,
    and standardized residuals from an estimated model.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.
    returns_series : pandas.Series
        The return series used in estimation.

    Returns
    -------
    fitted : pandas.DataFrame
        DataFrame containing fitted quantities for one model.
    """

    residuals = result.resid
    conditional_volatility = result.conditional_volatility
    conditional_variance = conditional_volatility ** 2
    standardized_residuals = residuals / conditional_volatility

    fitted = pd.DataFrame({
        "return": returns_series,
        "residual": residuals,
        "conditional_volatility": conditional_volatility,
        "conditional_variance": conditional_variance,
        "standardized_residual": standardized_residuals
    })

    fitted = fitted.dropna()

    return fitted

In [43]:
def run_residual_diagnostics(
    result,
    fitted_series,
    diagnostic_lags=(10, 20, 30),
    arch_lm_lags=10
):
    """
    Run Ljung-Box tests and ARCH-LM test on standardized residuals.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.
    fitted_series : pandas.DataFrame
        Output from extract_fitted_series().
    diagnostic_lags : tuple of int
        Lags used for Ljung-Box tests.
    arch_lm_lags : int
        Number of lags used in the ARCH-LM test.

    Returns
    -------
    diagnostics : dict
        Dictionary containing Ljung-Box and ARCH-LM diagnostic results.
    """

    std_resid = fitted_series["standardized_residual"].dropna()
    std_resid_sq = std_resid ** 2

    # Ljung-Box test on standardized residuals.
    lb_std_resid = acorr_ljungbox(
        std_resid,
        lags=list(diagnostic_lags),
        return_df=True
    )

    lb_std_resid = lb_std_resid.rename(columns={
        "lb_stat": "lb_stat_standardized_residual",
        "lb_pvalue": "lb_pvalue_standardized_residual"
    })

    # Ljung-Box test on squared standardized residuals.
    lb_std_resid_sq = acorr_ljungbox(
        std_resid_sq,
        lags=list(diagnostic_lags),
        return_df=True
    )

    lb_std_resid_sq = lb_std_resid_sq.rename(columns={
        "lb_stat": "lb_stat_squared_standardized_residual",
        "lb_pvalue": "lb_pvalue_squared_standardized_residual"
    })

    ljung_box_table = pd.concat(
        [lb_std_resid, lb_std_resid_sq],
        axis=1
    )

    ljung_box_table.index.name = "lag"
    ljung_box_table.insert(0, "model", result.model_name)
    ljung_box_table.insert(1, "distribution", result.distribution_name)

    # ARCH-LM test for remaining ARCH effects.
    arch_lm_stat, arch_lm_pvalue, arch_lm_f_stat, arch_lm_f_pvalue = het_arch(
        std_resid,
        nlags=arch_lm_lags,
        ddof=result.num_params
    )

    arch_lm_table = pd.Series({
        "model": result.model_name,
        "distribution": result.distribution_name,
        "lags": arch_lm_lags,
        "lm_stat": arch_lm_stat,
        "lm_pvalue": arch_lm_pvalue,
        "f_stat": arch_lm_f_stat,
        "f_pvalue": arch_lm_f_pvalue
    })

    diagnostics = {
        "ljung_box": ljung_box_table,
        "arch_lm": arch_lm_table
    }

    return diagnostics

In [44]:
test_result = fit_volatility_model(
    returns_series=returns_in,
    model_name="ARCH(1)",
    volatility_model="ARCH",
    p=1,
    o=0,
    q=0,
    distribution="normal"
)

test_parameter_table = extract_parameter_table(test_result)
test_model_info = extract_model_info(test_result)
test_fitted_series = extract_fitted_series(test_result, returns_in)
test_diagnostics = run_residual_diagnostics(
    result=test_result,
    fitted_series=test_fitted_series,
    diagnostic_lags=DIAGNOSTIC_LAGS,
    arch_lm_lags=ARCH_LM_LAGS
)

print(test_model_info)
print()
print(test_parameter_table)
print()
print(test_diagnostics["ljung_box"])
print()
print(test_diagnostics["arch_lm"])

model                    ARCH(1)
distribution              normal
log_likelihood      -8255.223415
aic                  16516.44683
bic                 16534.507642
n_obs                       3042
num_params                     3
convergence_flag               0
dtype: object

     model distribution parameter   estimate  std_error    t_value  \
0  ARCH(1)       normal        mu   0.185263   0.064805   2.858785   
1  ARCH(1)       normal     omega  11.671028   0.363500  32.107330   
2  ARCH(1)       normal  alpha[1]   0.164859   0.025733   6.406575   

         p_value  
0   4.252668e-03  
1  3.483574e-226  
2   1.488250e-10  

       model distribution  lb_stat_standardized_residual  \
lag                                                        
10   ARCH(1)       normal                      13.016149   
20   ARCH(1)       normal                      20.445164   
30   ARCH(1)       normal                      32.874936   

     lb_pvalue_standardized_residual  lb_stat_squared_standard